# FraudLens AI — Evaluation Notebook

**Team:** FraudLens AI | **Track:** AI Data Analyst Agent
**Purpose:** Evaluate the real rule-based fraud detector (`circular_transfer_detector.py` — circular transfers, velocity fraud, mule accounts, high-risk transfers) against a labeled transaction set, and reproduce the metrics reported in Section 9.

**No API key needed.** This detector is 100% rule-based (no ML, no LLM calls) — this notebook just imports it and runs it directly.

**Before running — one setup step:**
Place this notebook inside your repo (e.g. `FraudLens/notebooks/evaluation.ipynb`), so it sits alongside the `agents/` folder. The import cell below adds the project root to the path automatically, assuming this file is one folder below the root (adjust `PROJECT_ROOT` if your layout differs).


In [ ]:
import os
import sys
import json
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)

# --- make sure Python can find your agents/ folder ---
# This notebook is assumed to live at <repo_root>/notebooks/evaluation.ipynb.
# If you put it somewhere else, change PROJECT_ROOT to point at your repo root.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from agents.circular_transfer_detector import detect_fraud_patterns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 1. Build a Labeled Evaluation Set

No labeled dataset was supplied, so this cell **generates synthetic transactions with known ground truth** — a mix of:
- Clean accounts doing ordinary, unremarkable transfers (label: **not fraud**)
- Accounts deliberately placed in a circular transfer cycle (label: **fraud**)
- An account doing a rapid-fire velocity burst (label: **fraud**)
- A newly created mule account receiving a large sum quickly (label: **fraud**)
- An account sending one wildly-outsized transfer vs. its own history (label: **fraud**)

> **Replace this cell** with your real labeled transactions (and account metadata) to reproduce your actual Section 9 numbers. The detector just needs the same `transactions` / `accounts` schema shown in the module's docstring.


In [ ]:
def generate_labeled_dataset(seed=RANDOM_SEED, n_clean_accounts=15):
    rng = np.random.default_rng(seed)
    base_time = datetime(2026, 7, 1, 9, 0, 0)
    transactions = []
    accounts_meta = []
    ground_truth = {}  # account_id -> 1 (fraud) / 0 (not fraud)

    # --- clean accounts: normal, unremarkable transfers ---
    clean_ids = [f"ACC_CLEAN_{i}" for i in range(n_clean_accounts)]
    for acc in clean_ids:
        accounts_meta.append({
            "account_id": acc,
            "created_at": (base_time - timedelta(days=int(rng.integers(60, 800)))).isoformat()
        })
        ground_truth[acc] = 0
        n_txns = int(rng.integers(4, 9))
        for i in range(n_txns):
            receiver = clean_ids[int(rng.integers(0, len(clean_ids)))]
            if receiver == acc:
                continue
            transactions.append({
                "txn_id": f"CLEAN_{acc}_{i}",
                "sender": acc,
                "receiver": receiver,
                "amount": round(float(np.clip(rng.normal(300, 100), 20, 900)), 2),
                "timestamp": (base_time + timedelta(days=int(rng.integers(0, 30)),
                                                      hours=int(rng.integers(0, 24)))).isoformat()
            })

    # --- fraud pattern 1: circular transfer A -> B -> C -> A ---
    for acc in ["ACC_FRAUD_CYCLE_A", "ACC_FRAUD_CYCLE_B", "ACC_FRAUD_CYCLE_C"]:
        accounts_meta.append({"account_id": acc, "created_at": (base_time - timedelta(days=200)).isoformat()})
        ground_truth[acc] = 1
    transactions += [
        {"txn_id": "CYC1", "sender": "ACC_FRAUD_CYCLE_A", "receiver": "ACC_FRAUD_CYCLE_B",
         "amount": 250000, "timestamp": "2026-07-05T09:00:00"},
        {"txn_id": "CYC2", "sender": "ACC_FRAUD_CYCLE_B", "receiver": "ACC_FRAUD_CYCLE_C",
         "amount": 240000, "timestamp": "2026-07-05T09:10:00"},
        {"txn_id": "CYC3", "sender": "ACC_FRAUD_CYCLE_C", "receiver": "ACC_FRAUD_CYCLE_A",
         "amount": 235000, "timestamp": "2026-07-05T09:20:00"},
    ]

    # --- fraud pattern 2: velocity burst ---
    acc = "ACC_FRAUD_VELOCITY"
    accounts_meta.append({"account_id": acc, "created_at": (base_time - timedelta(days=100)).isoformat()})
    ground_truth[acc] = 1
    for i in range(7):
        transactions.append({
            "txn_id": f"VEL_{i}", "sender": acc, "receiver": f"ACC_CLEAN_{i % n_clean_accounts}",
            "amount": 1000 + i * 50,
            "timestamp": f"2026-07-06T09:0{i}:00" if i < 10 else f"2026-07-06T09:{i}:00"
        })

    # --- fraud pattern 3: mule account ---
    acc = "ACC_FRAUD_MULE"
    accounts_meta.append({"account_id": acc, "created_at": "2026-07-07T00:00:00"})
    ground_truth[acc] = 1
    transactions += [
        {"txn_id": "MULE1", "sender": "ACC_CLEAN_0", "receiver": acc,
         "amount": 90000, "timestamp": "2026-07-08T00:00:00"},
        {"txn_id": "MULE2", "sender": "ACC_CLEAN_1", "receiver": acc,
         "amount": 60000, "timestamp": "2026-07-09T00:00:00"},
    ]

    # --- fraud pattern 4: high-risk outlier transfer ---
    acc = "ACC_FRAUD_OUTLIER"
    accounts_meta.append({"account_id": acc, "created_at": (base_time - timedelta(days=150)).isoformat()})
    ground_truth[acc] = 1
    for i, amt in enumerate([1000, 1100, 950, 1050]):
        transactions.append({
            "txn_id": f"HR_NORMAL_{i}", "sender": acc, "receiver": f"ACC_CLEAN_{i % n_clean_accounts}",
            "amount": amt, "timestamp": f"2026-07-1{i}T09:00:00"
        })
    transactions.append({
        "txn_id": "HR_OUTLIER", "sender": acc, "receiver": "ACC_CLEAN_5",
        "amount": 50000, "timestamp": "2026-07-15T09:00:00"
    })

    return transactions, accounts_meta, ground_truth


transactions, accounts_meta, ground_truth = generate_labeled_dataset()
print(f"{len(transactions)} transactions across {len(ground_truth)} labeled accounts "
      f"({sum(ground_truth.values())} fraud, {len(ground_truth) - sum(ground_truth.values())} clean)")


## 2. Run the Real Detector

Calls `detect_fraud_patterns()` directly — the exact function your project uses — on the labeled transactions.


In [ ]:
detection_result = detect_fraud_patterns(transactions, accounts=accounts_meta)
print(f"mule_check_skipped: {detection_result['mule_check_skipped']}")

# every labeled account gets a score; accounts the detector never flagged get 0
rows = []
for acc, true_label in ground_truth.items():
    info = detection_result["accounts"].get(acc, {"risk_score": 0, "rules_fired": []})
    rows.append({
        "account": acc,
        "true_fraud": true_label,
        "risk_score": info["risk_score"],
        "rules_fired": ", ".join(info["rules_fired"]) if info["rules_fired"] else "-"
    })

results_df = pd.DataFrame(rows)
results_df.to_csv("fraudlens_eval_results.csv", index=False)
results_df.sort_values("risk_score", ascending=False)


## 3. Evaluation Metrics (Section 9)

A `risk_score` needs a cutoff to become a yes/no fraud call. `RISK_THRESHOLD` below is the operating threshold — change it to match whatever threshold your project actually uses in production (if any); otherwise 1 (any rule fired at all) is a reasonable default to start from.


In [ ]:
RISK_THRESHOLD = 1  # any nonzero score counts as "flagged" — adjust to match your project's real cutoff

y_true = results_df["true_fraud"].astype(int)
y_score = results_df["risk_score"].astype(float)
y_pred = (y_score >= RISK_THRESHOLD).astype(int)

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_true, y_score)
pr_auc = average_precision_score(y_true, y_score)

print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")
print(f"ROC-AUC:   {roc_auc:.3f}")
print(f"PR-AUC:    {pr_auc:.3f}")
print()
print(classification_report(y_true, y_pred, target_names=["Legit", "Fraud"], zero_division=0))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — FraudLens Rule-Based Detector")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_score)
prec, rec, _ = precision_recall_curve(y_true, y_score)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()

axes[1].plot(rec, prec, label=f"PR-AUC = {pr_auc:.3f}", color="darkorange")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

plt.tight_layout()
plt.savefig("roc_pr_curves.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(data=results_df, x="risk_score", hue="true_fraud",
             bins=10, kde=False, palette={0: "steelblue", 1: "crimson"},
             element="step", stat="count", common_norm=False)
plt.xlabel("Risk Score (0-100)")
plt.title("Risk Score Distribution: Legit vs Fraud accounts")
plt.legend(title="Actual Class", labels=["Fraud", "Legit"])
plt.tight_layout()
plt.savefig("risk_score_distribution.png", dpi=150)
plt.show()


## 4. Summary

Results are saved to `fraudlens_eval_results.csv`, and the three plots (`confusion_matrix.png`, `roc_pr_curves.png`, `risk_score_distribution.png`) are saved alongside this notebook — link/embed these in Section 9 of the report.

**Notes / next steps:**
- Swap the synthetic dataset in Section 1 for your real, labeled transactions + account metadata to get your actual reportable numbers.
- `RISK_THRESHOLD` in Section 3 determines what counts as "flagged" — set it to whatever cutoff your app actually uses (or sweep a few values and report the trade-off if you don't have one fixed yet).
- If your system combines this rule-based detector with another AI component (e.g. an LLM-based agent) before producing a final verdict, that combined pipeline should be evaluated separately, or this notebook extended to call it too.
